# Analysis: Identify cell types in the left and right hemisphere datasets through clustering and marker gene analysis.

- **[1]** Load both datasets (1695_Left and 1695_Right) and perform quality control by examining nCount_Spatial and nFeature_Spatial distributions to identify and filter low-quality cells. → `fig_01_qc_distributions.png`
- **[2]** Normalize and log-transform gene expression data, then identify highly variable genes for downstream analysis.
- **[3]** Perform PCA on highly variable genes to reduce dimensionality.
- **[4]** Compute neighborhood graph and perform UMAP dimensionality reduction for visualization. → `fig_02_umap_unclustered.png`
- **[5]** Cluster cells using Leiden algorithm at multiple resolutions to identify cell populations. → `fig_03_umap_clustered.png`
- **[6]** Rank genes differentially expressed between clusters using rank_genes_groups to identify cluster-specific markers.
- **[7]** Visualize top marker genes across clusters using dotplot and heatmap to infer cell type identities. → `fig_04_marker_genes.png`
- **[8]** Annotate clusters with inferred cell types based on known biological markers and create final annotated UMAP. → `fig_05_umap_annotated.png`

In [ ]:
import scanpy as sc
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

In [ ]:
DATA_FILES = ['/Users/lele/Library/Mobile Documents/com~apple~CloudDocs/Research/CWM/code/Agentic_toy_example/data/1695_Left.h5ad', '/Users/lele/Library/Mobile Documents/com~apple~CloudDocs/Research/CWM/code/Agentic_toy_example/data/1695_Right.h5ad']
RESULTS_DIR = "results/"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scanpy as sc
import warnings
warnings.filterwarnings('ignore')

In [ ]:
sc.settings.verbosity = 1

In [ ]:
print("=== Step 1: Load datasets and perform QC ===")

In [ ]:
adata_left = sc.read_h5ad(DATA_FILES[0])
adata_right = sc.read_h5ad(DATA_FILES[1])

In [ ]:
adata_left.obs['sample'] = '1695_Left'
adata_right.obs['sample'] = '1695_Right'

In [ ]:
print(f"Left dataset: {adata_left.shape[0]} cells, {adata_left.shape[1]} genes")
print(f"Right dataset: {adata_right.shape[0]} cells, {adata_right.shape[1]} genes")

In [ ]:
# QC distributions figure
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('QC Distributions - Left and Right Hemispheres', fontsize=14, fontweight='bold')

In [ ]:
axes[0, 0].hist(adata_left.obs['nCount_Spatial'], bins=50, color='steelblue', edgecolor='black', linewidth=0.5)
axes[0, 0].set_title('1695_Left: nCount_Spatial')
axes[0, 0].set_xlabel('Total Counts')
axes[0, 0].set_ylabel('Number of Spots')
axes[0, 0].axvline(adata_left.obs['nCount_Spatial'].quantile(0.05), color='red', linestyle='--', label='5th percentile')
axes[0, 0].legend()

In [ ]:
axes[0, 1].hist(adata_left.obs['nFeature_Spatial'], bins=50, color='steelblue', edgecolor='black', linewidth=0.5)
axes[0, 1].set_title('1695_Left: nFeature_Spatial')
axes[0, 1].set_xlabel('Number of Genes Detected')
axes[0, 1].set_ylabel('Number of Spots')
axes[0, 1].axvline(adata_left.obs['nFeature_Spatial'].quantile(0.05), color='red', linestyle='--', label='5th percentile')
axes[0, 1].legend()

In [ ]:
axes[1, 0].hist(adata_right.obs['nCount_Spatial'], bins=50, color='coral', edgecolor='black', linewidth=0.5)
axes[1, 0].set_title('1695_Right: nCount_Spatial')
axes[1, 0].set_xlabel('Total Counts')
axes[1, 0].set_ylabel('Number of Spots')
axes[1, 0].axvline(adata_right.obs['nCount_Spatial'].quantile(0.05), color='red', linestyle='--', label='5th percentile')
axes[1, 0].legend()

In [ ]:
axes[1, 1].hist(adata_right.obs['nFeature_Spatial'], bins=50, color='coral', edgecolor='black', linewidth=0.5)
axes[1, 1].set_title('1695_Right: nFeature_Spatial')
axes[1, 1].set_xlabel('Number of Genes Detected')
axes[1, 1].set_ylabel('Number of Spots')
axes[1, 1].axvline(adata_right.obs['nFeature_Spatial'].quantile(0.05), color='red', linestyle='--', label='5th percentile')
axes[1, 1].legend()

In [ ]:
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig_01_qc_distributions.png"), bbox_inches='tight', dpi=150)
plt.close()
print("Saved fig_01_qc_distributions.png")

In [ ]:
# Filter low-quality cells (bottom 5% by nCount_Spatial)
left_min_counts = adata_left.obs['nCount_Spatial'].quantile(0.05)
right_min_counts = adata_right.obs['nCount_Spatial'].quantile(0.05)
left_min_genes = adata_left.obs['nFeature_Spatial'].quantile(0.05)
right_min_genes = adata_right.obs['nFeature_Spatial'].quantile(0.05)

In [ ]:
adata_left = adata_left[
    (adata_left.obs['nCount_Spatial'] >= left_min_counts) &
    (adata_left.obs['nFeature_Spatial'] >= left_min_genes)
].copy()
adata_right = adata_right[
    (adata_right.obs['nCount_Spatial'] >= right_min_counts) &
    (adata_right.obs['nFeature_Spatial'] >= right_min_genes)
].copy()

In [ ]:
print(f"After QC filtering - Left: {adata_left.shape[0]} cells, Right: {adata_right.shape[0]} cells")

In [ ]:
# Concatenate datasets
print("Concatenating datasets...")
adata = sc.concat(
    [adata_left, adata_right],
    label='sample',
    keys=['1695_Left', '1695_Right'],
    uns_merge='unique'
)
adata.obs_names_make_unique()
print(f"Combined dataset: {adata.shape[0]} cells, {adata.shape[1]} genes")

In [ ]:
print("\n=== Step 2: Normalize, log-transform, and find highly variable genes ===")

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key='sample')
print(f"Number of highly variable genes: {adata.var['highly_variable'].sum()}")

In [ ]:
print("\n=== Step 3: PCA ===")
sc.pp.pca(adata, n_comps=50, use_highly_variable=True)
print("PCA completed")

In [ ]:
print("\n=== Step 4: Neighborhood graph and UMAP ===")
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata)
print("UMAP completed")

In [ ]:
# UMAP colored by sample
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('UMAP - Unclustered', fontsize=14, fontweight='bold')

In [ ]:
sample_colors = {'1695_Left': 'steelblue', '1695_Right': 'coral'}
for sample, color in sample_colors.items():
    mask = adata.obs['sample'] == sample
    axes[0].scatter(
        adata.obsm['X_umap'][mask, 0],
        adata.obsm['X_umap'][mask, 1],
        c=color, s=8, alpha=0.6, label=sample
    )
axes[0].set_title('Colored by Sample')
axes[0].set_xlabel('UMAP1')
axes[0].set_ylabel('UMAP2')
axes[0].legend(markerscale=2)

In [ ]:
sc_counts = adata.obs['nCount_Spatial'].values
sc_scatter = axes[1].scatter(
    adata.obsm['X_umap'][:, 0],
    adata.obsm['X_umap'][:, 1],
    c=np.log1p(sc_counts), cmap='viridis', s=8, alpha=0.7
)
axes[1].set_title('Colored by log(nCount_Spatial)')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')
plt.colorbar(sc_scatter, ax=axes[1], label='log(nCount_Spatial)')

In [ ]:
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig_02_umap_unclustered.png"), bbox_inches='tight', dpi=150)
plt.close()
print("Saved fig_02_umap_unclustered.png")

In [ ]:
print("\n=== Step 5: Leiden clustering ===")
sc.tl.leiden(adata, resolution=0.5, key_added='leiden')
n_clusters = adata.obs['leiden'].nunique()
print(f"Leiden clustering (res=0.5): {n_clusters} clusters")

In [ ]:
# UMAP with leiden clusters
import matplotlib.cm as cm
leiden_cats = adata.obs['leiden'].cat.categories.tolist()
n_cats = len(leiden_cats)
palette = cm.get_cmap('tab20', n_cats)
color_map = {cat: palette(i) for i, cat in enumerate(leiden_cats)}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('UMAP - Leiden Clustered (resolution=0.5)', fontsize=14, fontweight='bold')

In [ ]:
for cat in leiden_cats:
    mask = adata.obs['leiden'] == cat
    axes[0].scatter(
        adata.obsm['X_umap'][mask, 0],
        adata.obsm['X_umap'][mask, 1],
        c=[color_map[cat]], s=8, alpha=0.7, label=f'Cluster {cat}'
    )
axes[0].set_title('All Spots Colored by Cluster')
axes[0].set_xlabel('UMAP1')
axes[0].set_ylabel('UMAP2')
axes[0].legend(markerscale=2, bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

In [ ]:
# Per-sample leiden
for i, (sample, color_s) in enumerate([('1695_Left', 'steelblue'), ('1695_Right', 'coral')]):
    mask_s = adata.obs['sample'] == sample
    for cat in leiden_cats:
        mask = mask_s & (adata.obs['leiden'] == cat)
        if mask.sum() > 0:
            axes[1].scatter(
                adata.obsm['X_umap'][mask, 0],
                adata.obsm['X_umap'][mask, 1],
                c=[color_map[cat]], s=8, alpha=0.5,
                marker='o' if sample == '1695_Left' else '^'
            )
axes[1].set_title('By Sample (circle=Left, triangle=Right)')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')

In [ ]:
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig_03_umap_clustered.png"), bbox_inches='tight', dpi=150)
plt.close()
print("Saved fig_03_umap_clustered.png")

In [ ]:
print("\n=== Step 6: Rank genes (marker identification) ===")
sc.tl.rank_genes_groups(adata, groupby='leiden', method='wilcoxon', key_added='rank_genes_groups')
print("Rank genes groups completed")

In [ ]:
print("\n=== Step 7: Visualize top marker genes ===")

In [ ]:
# Get top markers per cluster
n_top = 5
top_markers = {}
for cluster in leiden_cats:
    try:
        genes = sc.get.rank_genes_groups_df(adata, group=cluster, key='rank_genes_groups')
        genes = genes[genes['pvals_adj'] < 0.05].head(n_top)['names'].tolist()
        top_markers[cluster] = genes
    except Exception:
        top_markers[cluster] = []

In [ ]:
all_markers = []
seen = set()
for cluster in leiden_cats:
    for g in top_markers[cluster]:
        if g not in seen:
            all_markers.append(g)
            seen.add(g)

In [ ]:
# Limit markers for visualization
if len(all_markers) > 60:
    all_markers = all_markers[:60]

In [ ]:
print(f"Total unique marker genes for visualization: {len(all_markers)}")

In [ ]:
# Dotplot
try:
    fig_dot, ax_dot = plt.subplots(figsize=(max(14, len(all_markers) * 0.25), max(5, n_cats * 0.5 + 2)))
    sc.pl.dotplot(
        adata,
        var_names=all_markers,
        groupby='leiden',
        ax=ax_dot,
        show=False,
        return_fig=False
    )
    plt.suptitle('Top Marker Genes per Leiden Cluster (Dotplot)', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "fig_04_marker_genes.png"), bbox_inches='tight', dpi=150)
    plt.close()
    print("Saved fig_04_marker_genes.png (dotplot)")
except Exception as e:
    print(f"Dotplot failed ({e}), using heatmap instead")
    # Fallback: compute mean expression per cluster for top markers
    adata_sub = adata[:, [g for g in all_markers if g in adata.var_names]].copy()
    cluster_means = pd.DataFrame(
        index=leiden_cats,
        columns=[g for g in all_markers if g in adata.var_names],
        dtype=float
    )
    for cat in leiden_cats:
        mask = adata.obs['leiden'] == cat
        cluster_means.loc[cat] = adata_sub.X[mask].mean(axis=0).A1 if hasattr(adata_sub.X, 'A1') else adata_sub.X[mask].mean(axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(max(14, len(all_markers) * 0.25), max(5, n_cats * 0.4 + 2)))
    im = ax.imshow(cluster_means.values.astype(float), aspect='auto', cmap='RdBu_r')
    ax.set_xticks(range(len(cluster_means.columns)))
    ax.set_xticklabels(cluster_means.columns, rotation=90, fontsize=7)
    ax.set_yticks(range(len(cluster_means.index)))
    ax.set_yticklabels([f'Cluster {c}' for c in cluster_means.index])
    ax.set_title('Mean Expression of Top Marker Genes per Cluster')
    plt.colorbar(im, ax=ax, label='Mean log-normalized expression')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "fig_04_marker_genes.png"), bbox_inches='tight', dpi=150)
    plt.close()
    print("Saved fig_04_marker_genes.png (heatmap fallback)")

In [ ]:
print("\n=== Step 8: Annotate clusters with cell types ===")

In [ ]:
# Inspect top markers per cluster and assign cell types
# Known brain cell type markers:
# Neurons: Snap25, Rbfox3, Syt1, Tubb3, Map2, Nefl, Nefm, Nefh, Neurod1, Rbfox1
# Excitatory neurons: Slc17a7 (Vglut1), Slc17a6 (Vglut2), Camk2a, Satb2
# Inhibitory neurons: Gad1, Gad2, Slc32a1 (Vgat)
# Astrocytes: Gfap, Aqp4, Aldh1l1, S100b, Slc1a2, Slc1a3
# Oligodendrocytes: Mbp, Plp1, Mog, Mobp, Cnp, Cldn11
# OPCs: Pdgfra, Cspg4, Olig2, Sox10
# Microglia: Cx3cr1, P2ry12, Tmem119, Csf1r, Iba1, Aif1
# Endothelial: Pecam1, Cldn5, Tie1, Flt1, Tek
# Pericytes: Pdgfrb, Rgs5, Notch3, Acta2

In [ ]:
known_markers = {
    'Excitatory_Neuron': ['Slc17a7', 'Slc17a6', 'Camk2a', 'Satb2', 'Neurod2', 'Snap25', 'Syt1'],
    'Inhibitory_Neuron': ['Gad1', 'Gad2', 'Slc32a1', 'Snap25', 'Syt1'],
    'Astrocyte': ['Gfap', 'Aqp4', 'Aldh1l1', 'S100b', 'Slc1a2', 'Slc1a3'],
    'Oligodendrocyte': ['Mbp', 'Plp1', 'Mog', 'Mobp', 'Cnp', 'Cldn11'],
    'OPC': ['Pdgfra', 'Cspg4', 'Olig2', 'Sox10'],
    'Microglia': ['Cx3cr1', 'P2ry12', 'Tmem119', 'Csf1r', 'Aif1'],
    'Endothelial': ['Pecam1', 'Cldn5', 'Flt1', 'Tie1'],
    'Pericyte': ['Pdgfrb', 'Rgs5', 'Notch3', 'Acta2'],
    'Neuron': ['Snap25', 'Rbfox3', 'Syt1', 'Tubb3', 'Map2', 'Nefl', 'Nefm'],
}

In [ ]:
# Score each cluster against known markers
cluster_scores = pd.DataFrame(0.0, index=leiden_cats, columns=list(known_markers.keys()))

In [ ]:
for cell_type, markers in known_markers.items():
    available = [m for m in markers if m in adata.var_names]
    if available:
        sc.tl.score_genes(adata, gene_list=available, score_name=f'score_{cell_type}', use_raw=False)
        for cat in leiden_cats:
            mask = adata.obs['leiden'] == cat
            cluster_scores.loc[cat, cell_type] = adata.obs.loc[mask, f'score_{cell_type}'].mean()

In [ ]:
print("\nCluster cell type scores:")
print(cluster_scores.round(3))

In [ ]:
# Assign cell type to each cluster by highest score
# Also check top markers for override
cell_type_assignments = {}
for cat in leiden_cats:
    best_type = cluster_scores.loc[cat].idxmax()
    best_score = cluster_scores.loc[cat].max()
    
    # Get top markers for this cluster
    top_genes = top_markers.get(cat, [])
    
    # Override logic based on top markers presence
    override = None
    for g in top_genes:
        g_lower = g.lower()
        if g in ['Mbp', 'Plp1', 'Mog', 'Mobp', 'Cnp'] or g_lower in ['mbp', 'plp1', 'mog', 'mobp', 'cnp']:
            override = 'Oligodendrocyte'
            break
        if g in ['Gfap', 'Aqp4', 'Aldh1l1', 'S100b'] or g_lower in ['gfap', 'aqp4', 'aldh1l1', 's100b']:
            override = 'Astrocyte'
            break
        if g in ['Gad1', 'Gad2'] or g_lower in ['gad1', 'gad2']:
            override = 'Inhibitory_Neuron'
            break
        if g in ['Slc17a7', 'Slc17a6'] or g_lower in ['slc17a7', 'slc17a6']:
            override = 'Excitatory_Neuron'
            break
        if g in ['Cx3cr1', 'P2ry12', 'Tmem119'] or g_lower in ['cx3cr1', 'p2ry12', 'tmem119']:
            override = 'Microglia'
            break
        if g in ['Pdgfra', 'Cspg4'] or g_lower in ['pdgfra', 'cspg4']:
            override = 'OPC'
            break
        if g in ['Pecam1', 'Cldn5'] or g_lower in ['pecam1', 'cldn5']:
            override = 'Endothelial'
            break
    
    assigned = override if override else best_type
    cell_type_assignments[cat] = assigned
    print(f"Cluster {cat}: {assigned} (score={best_score:.3f}, top genes: {top_genes[:3]})")

In [ ]:
# Map to adata
adata.obs['cell_type'] = adata.obs['leiden'].map(cell_type_assignments)
print("\nCell type distribution:")
print(adata.obs['cell_type'].value_counts())

In [ ]:
# Create annotated UMAP
unique_cell_types = adata.obs['cell_type'].unique().tolist()
n_types = len(unique_cell_types)
type_palette = cm.get_cmap('tab10', n_types)
type_color_map = {ct: type_palette(i) for i, ct in enumerate(unique_cell_types)}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('UMAP - Cell Type Annotated', fontsize=14, fontweight='bold')

In [ ]:
# Left: annotated by cell type
for ct in unique_cell_types:
    mask = adata.obs['cell_type'] == ct
    axes[0].scatter(
        adata.obsm['X_umap'][mask, 0],
        adata.obsm['X_umap'][mask, 1],
        c=[type_color_map[ct]], s=10, alpha=0.7, label=ct
    )
axes[0].set_title('All Spots - Cell Type Annotation')
axes[0].set_xlabel('UMAP1')
axes[0].set_ylabel('UMAP2')
axes[0].legend(markerscale=2, bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)

In [ ]:
# Right: by sample with cell type annotation
for sample, marker_style in [('1695_Left', 'o'), ('1695_Right', '^')]:
    mask_s = adata.obs['sample'] == sample
    for ct in unique_cell_types:
        mask = mask_s & (adata.obs['cell_type'] == ct)
        if mask.sum() > 0:
            axes[1].scatter(
                adata.obsm['X_umap'][mask, 0],
                adata.obsm['X_umap'][mask, 1],
                c=[type_color_map[ct]], s=10, alpha=0.5, marker=marker_style
            )

In [ ]:
# Add legend entries for samples
from matplotlib.lines import Line2D
sample_legend = [
    Line2D([0], [0], marker='o', color='gray', markersize=8, label='1695_Left', linestyle='None'),
    Line2D([0], [0], marker='^', color='gray', markersize=8, label='1695_Right', linestyle='None'),
]
ct_legend = [
    Line2D([0], [0], marker='o', color=type_color_map[ct], markersize=8, label=ct, linestyle='None')
    for ct in unique_cell_types
]
axes[1].legend(handles=sample_legend + ct_legend, bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
axes[1].set_title('By Sample & Cell Type (o=Left, ^=Right)')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')

In [ ]:
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig_05_umap_annotated.png"), bbox_inches='tight', dpi=150)
plt.close()
print("Saved fig_05_umap_annotated.png")

In [ ]:
print("\n=== Analysis Complete ===")
print(f"All figures saved to: {RESULTS_DIR}")
print("\nFinal cell type composition:")
for sample in ['1695_Left', '1695_Right']:
    print(f"\n{sample}:")
    mask = adata.obs['sample'] == sample
    print(adata.obs.loc[mask, 'cell_type'].value_counts().to_string())